# 02 — Comparing Models

Train multiple models on the same task and produce a comparison table.
Useful for deciding which model to deploy in an enterprise pipeline.

In [ ]:
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset
from pyhealth_enterprise.models.registry import ModelName, get_model
from pyhealth_enterprise.evaluation.metrics import compare_models, evaluate_binary_classifier
from pyhealth.tasks import readmission_prediction_mimic3_fn
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.trainer import Trainer
import pandas as pd

ds = SyntheticEHRDataset(); ds.load()
task_dataset = ds.dataset.set_task(readmission_prediction_mimic3_fn)
train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

In [ ]:
results = {}
for name in [ModelName.RETAIN, ModelName.TRANSFORMER]:
    model = get_model(name, task_dataset, ['conditions', 'drugs'], 'readmission')
    trainer = Trainer(model=model, metrics=['pr_auc', 'roc_auc', 'f1'])
    trainer.train(train_dataloader=train_loader, val_dataloader=val_loader,
                  epochs=20, monitor='pr_auc')
    r = trainer.evaluate(test_loader)
    results[name.value] = evaluate_binary_classifier(r['y_true'], r['y_prob'])

comparison = compare_models(results)
print(comparison.round(4))